# 🎙️ Kritix Voice Intelligence Engine (KVIE) — Colab GPU Studio

Run KVIE's upgraded speech-to-text inference engine, anti-hallucination guardrails, 4-tier grammar polisher, and FastAPI WebSocket server on a **free cloud NVIDIA GPU (T4 / A100)**.

### What this updated notebook provides:
1. **⚡ Faster-Whisper Large-v3-Turbo on CUDA**: Transcribes speech chunks in **~120–180 ms** using GPU Tensor Cores with greedy decoding (`temperature=0.0`).
2. **🛡️ Anti-Hallucination & Subtitle Blacklist**: Eliminates web/YouTube hallucination loops on silence with strict VAD energy gating and sanitized hotword prompts.
3. **✨ 4-Tier Multi-Tier Grammar Polisher (`GrammarRouter`)**: Automatically corrects typos, brand casing (`Kritix`, `GitHub`, `VS Code`), phonetic articles (`a university` / `an hour`), modal verbs, and removes verbal disfluencies (`um`, `uh`).
4. **🌐 Remote Tunnel to Desktop KVIE**: Expose the backend via Cloudflare Tunnel so your local desktop KVIE app connects seamlessly to this cloud GPU with live VRAM telemetry!

## 1. Verify GPU Acceleration & Hardware Specs

In [ ]:
# Check NVIDIA GPU allocation in Colab
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU: {gpu_name}")
    print(f"VRAM: {total_mem:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Please go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

## 2. Install KVIE & Updated Logic
This step installs all required CUDA packages, sets up CTranslate2 library paths, installs `cloudflared`, and unpacks the latest updated KVIE backend logic.

In [ ]:
# 1. Clone repository (or pull latest changes)
!git clone https://github.com/Aryan-sourcee/KVIE.git 2>/dev/null || (cd KVIE && git pull)
%cd KVIE

# 2. Check for uploaded update bundle (if you uploaded kvie_backend_update.zip to Colab)
import os, zipfile
for bundle_path in ["../kvie_backend_update.zip", "kvie_backend_update.zip", "/content/kvie_backend_update.zip", "dist_colab/kvie_backend_update.zip"]:
    if os.path.exists(bundle_path):
        print(f"📦 Found update bundle '{bundle_path}'. Unpacking updated KVIE logic...")
        with zipfile.ZipFile(bundle_path, "r") as zf:
            zf.extractall(".")
        break

# 3. Run automated installer (installs packages, configures CUDA 12 paths, installs cloudflared)
!bash install_colab.sh

## 3. Hugging Face Login (Optional for Gated Models)
If you plan to use gated models like `kritix-ai/Swar-stt`, enter your Hugging Face Token below. Otherwise, public models like `Systran/faster-whisper-large-v3-turbo` work instantly without a token.

In [ ]:
import os
from huggingface_hub import login

# Paste your HF token here if using gated models
HF_TOKEN = ""  # @param {type:"string"}

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip())
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("✅ Hugging Face authentication complete!")
else:
    print("ℹ️ No token provided. Public models (large-v3-turbo, small, tiny) will run directly!")

## 4. Test GPU STT Inference on Sample Speech
Benchmark Faster-Whisper on CUDA with greedy decoding (`temperature=0.0`) and anti-hallucination settings.

In [ ]:
import time
import numpy as np
from faster_whisper import WhisperModel

# Default: Systran/faster-whisper-large-v3-turbo (highest accuracy, near-realtime on T4 GPU)
MODEL_NAME = "Systran/faster-whisper-large-v3-turbo"  # @param ["Systran/faster-whisper-large-v3-turbo", "Systran/faster-whisper-small", "Systran/faster-whisper-tiny", "kritix-ai/Swar-stt"]

print(f"⏳ Loading model '{MODEL_NAME}' onto CUDA GPU with float16 precision...")
t0 = time.time()
model = WhisperModel(MODEL_NAME, device="cuda", compute_type="float16")
print(f"✅ Model loaded in {time.time() - t0:.2f} seconds!")

# Benchmark 0.5s dummy audio with greedy anti-hallucination parameters
dummy_audio = np.zeros(16000, dtype=np.float32)
t_inf = time.time()
segments, _ = model.transcribe(
    dummy_audio,
    beam_size=1,
    temperature=0.0,
    repetition_penalty=1.15,
    without_timestamps=False,
    hallucination_silence_threshold=1.5
)
list(segments)
print(f"⚡ CUDA Greedy Inference Warmup Latency: {(time.time() - t_inf)*1000:.1f} ms")

## 5. In-Browser Microphone Recording & Live 4-Tier Grammar Polisher
Speak into your laptop mic, click **Stop Recording**, and observe:
1. **Raw Whisper Transcription** on Colab GPU.
2. **Deterministic 4-Tier Grammar Polishing** (`TokenEngine` ➔ `SentenceEngine` ➔ `ParagraphEngine` ➔ `FinalEditor`).

In [ ]:
from IPython.display import HTML, Javascript, display
from google.colab import output
import base64
import io
import soundfile as sf
import librosa
from Backend.kvie.grammar_router import GrammarRouter

RECORD_HTML = """
<div style="font-family: sans-serif; padding: 18px; border: 1px solid #3f3f46; border-radius: 12px; background: #18181b; color: #f4f4f5; max-width: 500px;">
  <h3 style="margin-top: 0; color: #22c55e;">🎙️ KVIE Colab Voice & Grammar Test</h3>
  <p style="font-size: 13px; color: #a1a1aa;">Record speech from your browser and polish through KVIE's multi-tier engine:</p>
  <button id="rec-btn" style="background: #22c55e; color: #000; font-weight: bold; border: none; padding: 10px 18px; border-radius: 8px; cursor: pointer;">🔴 Start Recording</button>
  <span id="status" style="margin-left: 12px; font-size: 13px; color: #71717a;">Idle</span>
</div>
<script>
var mediaRecorder;
var audioChunks = [];
var recBtn = document.getElementById('rec-btn');
var statusText = document.getElementById('status');

recBtn.onclick = async () => {
  if (!mediaRecorder || mediaRecorder.state === 'inactive') {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    mediaRecorder = new MediaRecorder(stream);
    audioChunks = [];
    mediaRecorder.ondataavailable = e => audioChunks.push(e.data);
    mediaRecorder.onstop = async () => {
      statusText.innerText = 'Processing audio on GPU & routing grammar...';
      const blob = new Blob(audioChunks, { type: 'audio/webm' });
      const reader = new FileReader();
      reader.readAsDataURL(blob);
      reader.onloadend = () => {
        google.colab.kernel.invokeFunction('notebook.transcribe_audio', [reader.result], {});
      };
    };
    mediaRecorder.start();
    recBtn.innerText = '⏹️ Stop Recording';
    recBtn.style.background = '#f43f5e';
    recBtn.style.color = '#fff';
    statusText.innerText = 'Listening... Speak now!';
  } else {
    mediaRecorder.stop();
    recBtn.innerText = '🔴 Start Recording';
    recBtn.style.background = '#22c55e';
    recBtn.style.color = '#000';
  }
};
</script>
"""

def transcribe_audio_callback(base64_audio):
    try:
        header, encoded = base64_audio.split(",", 1)
        data = base64.b64decode(encoded)
        audio_file = io.BytesIO(data)
        
        # Load and resample to 16 kHz
        samples, sr = librosa.load(audio_file, sr=16000, mono=True)
        duration = len(samples) / 16000.0
        
        t_start = time.time()
        segments, info = model.transcribe(
            samples,
            beam_size=1,
            language="en",
            temperature=0.0,
            repetition_penalty=1.15,
            without_timestamps=False,
            hallucination_silence_threshold=1.5
        )
        raw_text = " ".join(s.text.strip() for s in segments).strip()
        latency = (time.time() - t_start) * 1000
        
        # Polish with KVIE 4-Tier Grammar Engine
        polished_text = GrammarRouter.route(raw_text)
        
        print("\n" + "="*65)
        print(f"🎯 Transcribed ({duration:.1f}s audio in {latency:.0f} ms on CUDA GPU):")
        print(f"📝 Raw Acoustic:     \"{raw_text}\"")
        print(f"✨ Grammar Polished: \"{polished_text}\"")
        print("="*65 + "\n")
    except Exception as e:
        print(f"Error processing audio: {e}")

output.register_callback('notebook.transcribe_audio', transcribe_audio_callback)
display(HTML(RECORD_HTML))

## 6. Launch Remote KVIE FastAPI Backend & Cloudflare Tunnel
Run the cell below to launch the KVIE FastAPI backend on GPU port 8765 and create a public Cloudflare Tunnel.

Then copy the generated URL into **Desktop KVIE ➔ Settings ➔ Backend Service & Remote GPU**!

In [ ]:
import os
import time
import subprocess
import requests
from IPython.display import HTML, display

# 1. Start KVIE FastAPI backend in background on port 8765
env = os.environ.copy()
env["KVIE_HOST"] = "0.0.0.0"
env["KVIE_PORT"] = "8765"
env["KVIE_ALLOWED_ORIGINS"] = "*"

server_process = subprocess.Popen(
    ["python", "-m", "Backend.kvie.service"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Starting KVIE FastAPI Service and warming up GPU...")
time.sleep(5)

# Verify health endpoint
telemetry = {}
try:
    resp = requests.get("http://127.0.0.1:8765/health", timeout=5)
    if resp.status_code == 200:
        telemetry = resp.json()
        print("✅ Backend Health Verified:")
        print(f"   • Device: {telemetry.get('device_name', 'CUDA')}")
        print(f"   • Model: {telemetry.get('model_name', 'large-v3-turbo')}")
        print(f"   • Precision: {telemetry.get('compute_type', 'float16')}")
        if telemetry.get('vram_total_mb'):
            print(f"   • VRAM: {telemetry.get('vram_allocated_mb', 0):.0f} MB / {telemetry.get('vram_total_mb', 0):.0f} MB")
except Exception as e:
    print(f"ℹ️ Service starting: {e}")

# 2. Open Cloudflare Tunnel to expose port 8765 publicly
print("\n🌐 Opening public Cloudflare tunnel to port 8765...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8765"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part:
                public_url = part.strip()
                if not public_url.startswith("http"):
                    public_url = "https://" + public_url
                break
        if public_url:
            break

if public_url:
    display(HTML(f"""
    <div style="font-family: sans-serif; padding: 20px; border: 2px solid #22c55e; border-radius: 12px; background: #0f172a; color: #f8fafc; max-width: 650px; margin-top: 15px;">
      <h3 style="margin-top: 0; color: #22c55e; display: flex; align-items: center; gap: 8px;">
        🚀 Remote KVIE GPU Backend Active!
      </h3>
      <p style="color: #94a3b8; font-size: 14px; margin-bottom: 8px;">Copy your public Cloudflare Tunnel URL below:</p>
      <div style="background: #1e293b; padding: 12px; border-radius: 8px; font-family: monospace; font-size: 15px; color: #38bdf8; word-break: break-all; border: 1px solid #334155;">
        {public_url}
      </div>
      <h4 style="margin-top: 16px; margin-bottom: 8px; color: #f8fafc;">📱 Connect Desktop KVIE in 3 Clicks:</h4>
      <ol style="color: #cbd5e1; font-size: 13px; line-height: 1.6; padding-left: 20px; margin: 0;">
        <li>Open your Desktop KVIE App (installed via <code>KVIE_2.0.0_x64-setup.exe</code>)</li>
        <li>Click <b>Settings</b> (⚙️) ➔ Scroll to <b>Backend Service & Remote GPU</b></li>
        <li>Paste <code>{public_url}</code> into the URL field and click <b>Save Backend URL</b></li>
      </ol>
      <p style="margin-top: 12px; font-size: 12px; color: #64748b; margin-bottom: 0;">Status badge will turn green showing live remote GPU VRAM telemetry!</p>
    </div>
    """))
else:
    print("Tunnel started. Check output lines above for your trycloudflare.com URL.")